In [15]:
#import libraries
import cv2
import numpy as np
from PIL import Image
import os
import PIL
import glob
import imutils
from matplotlib import pyplot as plt 

#initialize x,y,w,h
x=0
y=0
w=0
h=0

# Load your image
image = cv2.imread('Front image/woman7.jpg')

def resize_with_aspect_ratio(image, width=None, height=None):
    h, w = image.shape[:2]

    if width is None and height is None:
        return image
    elif width is None:
        aspect_ratio = height / h
        new_width = int(w * aspect_ratio)
        new_height = height
    elif height is None:
        aspect_ratio = width / w
        new_width = width
        new_height = int(h * aspect_ratio)
    else:
        # Both width and height are specified, use the minimum aspect ratio
        aspect_ratio = min(width / w, height / h)
        new_width = int(w * aspect_ratio)
        new_height = int(h * aspect_ratio)

    resized_image = cv2.resize(image, (new_width, new_height))
    return resized_image

# Example usage
img1 = resize_with_aspect_ratio(image, width=400)  # Specify the desired width
cv2.imshow('Image', img1)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [16]:
img=cv2.cvtColor(img1,cv2.COLOR_RGB2GRAY) #convert colour from RGB to grayscale
cv2.imshow('Image', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [17]:
#for generating rect values
# Load the pre-trained HOG person detector
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# Detect person in the image
person, _ = hog.detectMultiScale(img)

# If person is detected, choose the largest bounding box as the main object
if len(person) > 0:
    # Get the largest bounding box
    x, y, w, h = max(person, key=lambda box: box[2] * box[3])
    rect = (x, y, w, h)
    print(f"Generated rect coordinates for the main object: {rect}")
else:
    print("No person detected in the image.")

No person detected in the image.


In [18]:
# Extracting the foreground using GrabCut method

# Create a mask (initialized with zeros)
mask = np.zeros(img.shape[:2], np.uint8)

# Initialize background and foreground models
bgdModel = np.zeros((1, 65), np.float64)
fgdModel = np.zeros((1, 65), np.float64)

# Apply GrabCut algorithm
cv2.grabCut(img1, mask, rect, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)

# Create a new mask (where pixels are either background or foreground)
mask2 = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')

# Multiply the original image with the mask to extract the foreground
foreground = cv2.bitwise_and(img, img, mask=mask2)

# Display the extracted foreground
cv2.imshow('Foreground', foreground)
cv2.waitKey(0)
cv2.destroyAllWindows()